# Apple (AAPL) News — Past-Year Collection

**Selenium + BeautifulSoup | Google News RSS | 2025-10-09 to 2026-10-09 (inclusive)**

Search month by month, validate publication timestamps, deduplicate articles, and export a FinBERT-ready CSV. **Google News RSS is not a complete historical archive**: empty or low-volume months are coverage gaps, not proof that no news existed. This notebook does not modify your original `News.csv`.

**Browser recovery edition:** a new Chrome session per month, one retry, and CSV checkpoints.

## 1. Setup
Install if needed: `python -m pip install selenium beautifulsoup4 pandas`. Use your `finbert` Jupyter kernel. Chrome must be installed.

In [7]:
from datetime import datetime, timedelta, timezone
from pathlib import Path
from urllib.parse import urlencode
import re
import time

import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait

START_DATE = pd.Timestamp("2025-10-09", tz="UTC")
END_EXCLUSIVE = pd.Timestamp("2026-10-10", tz="UTC")  # Includes Oct 9, 2026
QUERY = '"Apple Inc" OR "AAPL stock"'  # Match the previously successful query
HEADLESS = False
WAIT_SECONDS = 15
DELAY_SECONDS = 1.0

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTPUT_PATH = PROJECT_ROOT / "data" / "raw" / "aapl_news_2025_2026.csv"
COVERAGE_PATH = PROJECT_ROOT / "data" / "raw" / "aapl_news_monthly_coverage.csv"
print("Output:", OUTPUT_PATH)


Output: /Users/my/Desktop/TWO-LSTM/FinBert_LSTMGRU_GAN/finbert/data/raw/aapl_news_2025_2026.csv


## 2. Create monthly date windows
Google News uses `after:` / `before:` search operators. We still verify actual publication dates because search filtering can be imperfect. The last partial month is included.

In [8]:
def date_windows(start, end_exclusive):
    cursor = start
    while cursor < end_exclusive:
        next_month = (cursor + pd.offsets.MonthBegin(1)).normalize()
        stop = min(next_month, end_exclusive)
        yield cursor, stop
        cursor = stop

windows = list(date_windows(START_DATE, END_EXCLUSIVE))
for start, stop in windows:
    print(f"{start.date()} → {(stop - pd.Timedelta(days=1)).date()}")


2025-10-09 → 2025-10-31
2025-11-01 → 2025-11-30
2025-12-01 → 2025-12-31
2026-01-01 → 2026-01-31
2026-02-01 → 2026-02-28
2026-03-01 → 2026-03-31
2026-04-01 → 2026-04-30
2026-05-01 → 2026-05-31
2026-06-01 → 2026-06-30
2026-07-01 → 2026-07-31
2026-08-01 → 2026-08-31
2026-09-01 → 2026-09-30
2026-10-01 → 2026-10-09


## 3. Retrieve and parse RSS items
Each monthly query runs in a **fresh Chrome session** so a closed window cannot break subsequent months. Google News may limit results; counts are audited, not treated as a complete archive.

In [9]:
def make_rss_url(start, stop):
    dated_query = f"{QUERY} after:{start.date()} before:{stop.date()}"
    params = urlencode({"q": dated_query, "hl": "en-US", "gl": "US", "ceid": "US:en"})
    return f"https://news.google.com/rss/search?{params}"


def extract_link(tag):
    if tag is None:
        return ""
    for candidate in (tag.get("href"), tag.get_text(strip=True), str(tag.next_sibling).strip() if tag.next_sibling else ""):
        if candidate and candidate.startswith(("https://", "http://")):
            return candidate
    return ""


def parse_items(html, window_start, window_stop, rss_url):
    soup = BeautifulSoup(html, "html.parser")
    retrieved = datetime.now(timezone.utc).isoformat()
    rows = []
    for item in soup.select("item"):
        title_tag = item.find("title")
        pub_tag = item.find("pubdate")
        link_tag = item.find("link")
        source_tag = item.find("source")
        title = title_tag.get_text(" ", strip=True) if title_tag else ""
        if not title:
            continue
        rows.append({
            "Article Title": title,
            "Article URL": extract_link(link_tag),
            "Published At UTC": pub_tag.get_text(strip=True) if pub_tag else None,
            "Source": source_tag.get_text(" ", strip=True) if source_tag else "",
            "Retrieved At UTC": retrieved,
            "Search Window Start": str(window_start.date()),
            "Search Window End Exclusive": str(window_stop.date()),
            "RSS Search URL": rss_url,
        })
    return rows


## 4. Collect with browser recovery and checkpoints
Each month starts and closes its own Chrome browser. A failed month is retried once with a new browser. Results and coverage are checkpointed after every month, so a browser crash does not erase prior successful work.

**Important:** Use the same unparenthesized Apple query that worked in the browser. Google News may return at most 100 RSS items per search, and `after:/before:` searches are not guaranteed to be exhaustive. Check `Items Within Window` and the returned date range for each month; months with 100 results may require smaller weekly windows.

In [10]:
options = Options()
if HEADLESS:
    options.add_argument("--headless=new")

CHECKPOINT_NEWS = OUTPUT_PATH.with_name("aapl_news_checkpoint.csv")
CHECKPOINT_COVERAGE = COVERAGE_PATH.with_name("aapl_coverage_checkpoint.csv")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

all_rows = []
coverage_rows = []

for start, stop in windows:
    rss_url = make_rss_url(start, stop)
    rows = []
    error = ""

    for attempt in range(1, 3):
        driver = None
        try:
            # A fresh browser for each month/attempt avoids reusing a dead window.
            driver = webdriver.Chrome(options=options)
            driver.set_page_load_timeout(30)
            driver.get(rss_url)
            WebDriverWait(driver, WAIT_SECONDS).until(
                lambda d: d.execute_script("return document.readyState") == "complete"
            )
            rows = parse_items(driver.page_source, start, stop, rss_url)
            error = ""
            break
        except Exception as exc:
            error = f"{type(exc).__name__}: {str(exc).splitlines()[0]}"
            print(f"  {start.date()} attempt {attempt}/2 failed: {error}")
            time.sleep(2)
        finally:
            if driver is not None:
                try:
                    driver.quit()
                except Exception:
                    pass

    all_rows.extend(rows)
    dates = pd.to_datetime(
        pd.Series([r["Published At UTC"] for r in rows], dtype="object"),
        utc=True, errors="coerce"
    )
    in_window = dates.ge(start) & dates.lt(stop)
    coverage_rows.append({
        "Window Start": str(start.date()),
        "Window End Exclusive": str(stop.date()),
        "RSS Items": len(rows),
        "Valid Publication Dates": int(dates.notna().sum()),
        "Items Within Window": int(in_window.sum()),
        "First Returned Date": str(dates.min()) if dates.notna().any() else "",
        "Last Returned Date": str(dates.max()) if dates.notna().any() else "",
        "Hit 100 Item Limit": len(rows) >= 100,
        "Error": error,
    })

    # Save partial progress after every month.
    pd.DataFrame(all_rows).to_csv(CHECKPOINT_NEWS, index=False)
    pd.DataFrame(coverage_rows).to_csv(CHECKPOINT_COVERAGE, index=False)
    print(f"{start.date()} to {stop.date()}: items={len(rows)}, "
          f"in-window={int(in_window.sum())}, status={'ERROR' if error else 'OK'}")
    time.sleep(DELAY_SECONDS)

raw_news = pd.DataFrame(all_rows)
coverage = pd.DataFrame(coverage_rows)
print("Total RSS items (before deduplication):", len(raw_news))
display(coverage)
if coverage["Error"].astype(bool).any():
    print("WARNING: Some months failed. Review the Error column before treating the dataset as complete.")


2025-10-09 to 2025-11-01: items=31, in-window=31, status=OK
2025-11-01 to 2025-12-01: items=36, in-window=34, status=OK
2025-12-01 to 2026-01-01: items=47, in-window=47, status=OK
2026-01-01 to 2026-02-01: items=33, in-window=32, status=OK
2026-02-01 to 2026-03-01: items=38, in-window=38, status=OK
2026-03-01 to 2026-04-01: items=35, in-window=33, status=OK
2026-04-01 to 2026-05-01: items=73, in-window=68, status=OK
2026-05-01 to 2026-06-01: items=66, in-window=60, status=OK
2026-06-01 to 2026-07-01: items=92, in-window=86, status=OK
2026-07-01 to 2026-08-01: items=88, in-window=86, status=OK
2026-08-01 to 2026-09-01: items=80, in-window=72, status=OK
2026-09-01 to 2026-10-01: items=100, in-window=100, status=OK
2026-10-01 to 2026-10-10: items=100, in-window=100, status=OK
Total RSS items (before deduplication): 819


,Window Start,Window End Exclusive,RSS Items,Valid Publication Dates,Items Within Window,First Returned Date,Last Returned Date,Hit 100 Item Limit,Error
0,2025-10-09,2025-11-01,31,31,31,2025-10-10 07:00:00+00:00,2025-10-31 07:00:00+00:00,False,
1,2025-11-01,2025-12-01,36,36,34,2025-11-02 07:00:00+00:00,2025-12-01 08:00:00+00:00,False,
2,2025-12-01,2026-01-01,47,47,47,2025-12-01 08:00:00+00:00,2025-12-30 08:00:00+00:00,False,
3,2026-01-01,2026-02-01,33,33,32,2026-01-05 08:00:00+00:00,2026-02-01 08:00:00+00:00,False,
4,2026-02-01,2026-03-01,38,38,38,2026-02-01 08:00:00+00:00,2026-02-27 08:00:00+00:00,False,
5,2026-03-01,2026-04-01,35,35,33,2026-03-02 08:00:00+00:00,2026-04-01 07:00:00+00:00,False,
6,2026-04-01,2026-05-01,73,73,68,2026-04-01 07:00:00+00:00,2026-05-01 07:00:00+00:00,False,
7,2026-05-01,2026-06-01,66,60,60,2026-05-01 07:00:00+00:00,2026-05-31 07:00:00+00:00,False,
8,2026-06-01,2026-07-01,92,92,86,2026-06-01 07:00:00+00:00,2026-07-01 07:00:00+00:00,False,
9,2026-07-01,2026-08-01,88,88,86,2026-07-01 07:00:00+00:00,2026-08-01 07:00:00+00:00,False,


## 5. Clean, verify dates, and deduplicate
Filter by **actual publication timestamps**, not merely by the date query. Keep Google News redirect links as source references. Do not fill missing timestamps with scrape time.

In [11]:
if raw_news.empty:
    news = pd.DataFrame(columns=[
        "Article Title", "Article URL", "Published At UTC", "Source", "Retrieved At UTC",
        "Search Window Start", "Search Window End Exclusive", "RSS Search URL"
    ])
else:
    news = raw_news.copy()
    news["Published At UTC"] = pd.to_datetime(news["Published At UTC"], utc=True, errors="coerce")
    news["Article Title"] = news["Article Title"].fillna("").astype(str).str.strip()
    news["Article URL"] = news["Article URL"].fillna("").astype(str).str.strip()
    news = news.loc[
        news["Article Title"].ne("")
        & news["Article URL"].str.startswith(("https://", "http://"))
        & news["Published At UTC"].ge(START_DATE)
        & news["Published At UTC"].lt(END_EXCLUSIVE)
    ].copy()
    news = (news.drop_duplicates(subset=["Article URL"])
                .sort_values("Published At UTC")
                .reset_index(drop=True))

print("Unique dated articles:", len(news))
if not news.empty:
    print("Actual date range:", news["Published At UTC"].min(), "→", news["Published At UTC"].max())
display(news[["Article Title", "Article URL", "Published At UTC", "Source"]].head(10))


Unique dated articles: 795
Actual date range: 2025-10-10 07:00:00+00:00 → 2026-10-09 14:35:42+00:00


,Article Title,Article URL,Published At UTC,Source
0,Apple Accused of AI Copyright Infringement by ...,https://news.google.com/rss/articles/CBMinwFBV...,2025-10-10 07:00:00+00:00,
1,Apple sued over use of copyrighted books to tr...,https://news.google.com/rss/articles/CBMi1AFBV...,2025-10-10 07:00:00+00:00,
2,Key facts: Apple Inc. ranks in top 10 S&P 500 ...,https://news.google.com/rss/articles/CBMi4gFBV...,2025-10-11 07:00:00+00:00,
3,Neuroscientists Sue Apple Over Pirated 'Books3...,https://news.google.com/rss/articles/CBMiqAFBV...,2025-10-13 07:00:00+00:00,
4,Apple to Open iPhone Air Preorders in China on...,https://news.google.com/rss/articles/CBMipwFBV...,2025-10-13 07:00:00+00:00,
5,Apple expands renewable energy projects across...,https://news.google.com/rss/articles/CBMinAFBV...,2025-10-14 07:00:00+00:00,
6,M5 Chip Sets a New Benchmark in AI Performance...,https://news.google.com/rss/articles/CBMiZkFVX...,2025-10-15 07:00:00+00:00,
7,Apple introduces the powerful new iPad Pro wit...,https://news.google.com/rss/articles/CBMioAFBV...,2025-10-15 07:00:00+00:00,
8,Apple Vision Pro upgraded with the powerful M5...,https://news.google.com/rss/articles/CBMiowFBV...,2025-10-15 07:00:00+00:00,
9,"Apple unleashes M5, the next big leap in AI pe...",https://news.google.com/rss/articles/CBMitgFBV...,2025-10-15 07:00:00+00:00,


## 6. Coverage audit and export
A successful HTTP/page load is **not** evidence of complete historical coverage. Review counts and publication dates before running FinBERT. The monthly coverage file includes errors and empty months.

In [12]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
news.to_csv(OUTPUT_PATH, index=False)
coverage.to_csv(COVERAGE_PATH, index=False)

if not news.empty:
    monthly_counts = (news.set_index("Published At UTC")
                       .resample("MS").size().rename("Unique Articles"))
    display(monthly_counts.to_frame())
    print("Articles with empty URLs:", news["Article URL"].eq("").sum())
else:
    print("No valid historical articles returned. Check coverage and use an archival source if necessary.")

print("News CSV:", OUTPUT_PATH)
print("Coverage CSV:", COVERAGE_PATH)


,Unique Articles
Published At UTC,
2025-10-01 00:00:00+00:00,31
2025-11-01 00:00:00+00:00,34
2025-12-01 00:00:00+00:00,47
2026-01-01 00:00:00+00:00,32
2026-02-01 00:00:00+00:00,38
2026-03-01 00:00:00+00:00,33
2026-04-01 00:00:00+00:00,68
2026-05-01 00:00:00+00:00,61
2026-06-01 00:00:00+00:00,86


Articles with empty URLs: 0
News CSV: /Users/my/Desktop/TWO-LSTM/FinBert_LSTMGRU_GAN/finbert/data/raw/aapl_news_2025_2026.csv
Coverage CSV: /Users/my/Desktop/TWO-LSTM/FinBert_LSTMGRU_GAN/finbert/data/raw/aapl_news_monthly_coverage.csv


## 7. Next step: FinBERT and price alignment

- Run FinBERT on `Article Title`; retain per-article probabilities and publication timestamps.
- Group sentiment using an explicit trading-day availability rule (after-hours news should not leak into an earlier close).
- Retrieve AAPL price data for the same period; fit scalers on **training data only**.
- RSS results are **not** guaranteed to include every Apple article from the past year. If some months have little or no coverage, use a historical news archive before claiming a full-year study.
- Preserve the original 2010–2020 `News.csv`; this notebook creates a separate file.
